# 02 Zero-shot routing, a scoring head, and few-shot SetFit

Three ways to get a router with few or no labels, each tested against the same test split and judged by an interval:

1. **Zero-shot:** write a description of each route, embed descriptions and messages together, route to the nearest. No training.
2. **Scoring head:** a logistic regression on frozen embeddings, trained on 8, 32 or all labeled examples per route.
3. **SetFit:** fine-tune the embedding model itself with contrastive pairs built from 8 examples per route, then fit the head.

SetFit is accepted only if it beats a **prototype baseline** (the mean embedding of each route's 8 examples) over five seeds.

In [1]:
import json, time
import numpy as np
from pathlib import Path
import meridian as m
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression

RESULTS = Path("data/results"); RESULTS.mkdir(parents=True, exist_ok=True)
tr, ca, te = m.load("train"), m.load("calibration"), m.load("test")
_, ytr = m.texts_labels(tr); _, yca = m.texts_labels(ca); _, yte = m.texts_labels(te)
cl_te = m.cluster_ids(te)
E = m.cached_embeddings("bge-small")   # sentence-embedding model, 33M parameters
print("hardware:", m.hardware())

def report(name, pred, y=yte, clusters=cl_te):
    ok = (pred == y); k, n = int(ok.sum()), len(y)
    lo, hi = m.wilson(k, n); clo, chi = m.cluster_bootstrap_ci(ok.astype(float), clusters)
    print(f"{name:34s} acc {k/n:.3f}  Wilson [{lo:.3f}, {hi:.3f}]  cluster [{clo:.3f}, {chi:.3f}]")
    return dict(acc=k/n, k=k, n=n, wilson=[lo, hi], cluster_ci=[clo, chi])

out = {}

hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0


## 1. Zero-shot: route descriptions as tested artifacts
The descriptions are the model. A wording change moves accuracy, so they are versioned text and evaluated like code.

In [2]:
DESCRIPTIONS = {
    "baggage_claim": "A passenger reports luggage that is lost, delayed, damaged or missing from the belt and wants it traced or compensated.",
    "cancellation_request": "A customer asks to cancel a booking, flight, hotel or trip they have made.",
    "out_of_scope": "A message that is not about a booking or a travel problem: general chat, other businesses, or nothing a travel agent can act on.",
    "payment_dispute": "A customer disputes a charge, a double payment, an unrecognized transaction or a wrong amount taken from their card.",
    "rebooking_request": "A customer wants to change dates, times or flights on an existing booking, or be moved to another flight.",
    "refund_claim": "A customer asks for money back for something they paid for, such as a cancelled service, a missed flight or a fee.",
    "service_complaint": "A customer complains about rudeness, poor service, an unhelpful agent or a bad experience, without asking for a specific action.",
    "visa_entry_question": "A traveler asks about visa, passport, vaccination or entry requirements for a destination.",
}
from meridian import encode
D = encode("bge-small", [DESCRIPTIONS[r] for r in m.ROUTES])
sim_ca, sim_te = E["calibration"] @ D.T, E["test"] @ D.T
out["zero_shot_raw"] = report("zero-shot, cosine, no scaling", sim_te.argmax(1))

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


zero-shot, cosine, no scaling      acc 0.590  Wilson [0.521, 0.656]  cluster [0.450, 0.720]


### Temperature-scaled similarity
Raw cosines are not probabilities. One temperature, fitted on the calibration split by minimizing log loss, turns them into a distribution
that notebook 03 can calibrate and gate.

In [3]:
def nll(T, sim, y):
    p = softmax(sim / T, axis=1)
    return -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
grid = np.linspace(0.005, 0.2, 80)
T = grid[np.argmin([nll(t, sim_ca, yca) for t in grid])]
print(f"fitted temperature {T:.3f}; calibration log loss {nll(T, sim_ca, yca):.3f} (at T=0.05: {nll(0.05, sim_ca, yca):.3f})")
P_zero_te = softmax(sim_te / T, axis=1)
out["zero_shot"] = report("zero-shot, temperature-scaled", P_zero_te.argmax(1))
out["zero_shot_temperature"] = float(T)

fitted temperature 0.042; calibration log loss 1.427 (at T=0.05: 1.437)


zero-shot, temperature-scaled      acc 0.590  Wilson [0.521, 0.656]  cluster [0.450, 0.720]


### What does zero-shot confuse?
The three money-moving routes (refund, cancellation, rebooking) share vocabulary, so look at where the wrong ones go.

In [4]:
from sklearn.metrics import confusion_matrix
pred = P_zero_te.argmax(1)
cm = confusion_matrix(yte, pred, labels=range(8))
pairs = sorted(((cm[i, j], m.ROUTES[i], m.ROUTES[j]) for i in range(8) for j in range(8) if i != j), reverse=True)[:5]
for c, a, b in pairs: print(f"{c:3d} true {a:22s} -> predicted {b}")
out["zero_shot_top_confusions"] = [(int(c), a, b) for c, a, b in pairs]

 16 true out_of_scope           -> predicted visa_entry_question
 15 true refund_claim           -> predicted cancellation_request
 12 true service_complaint      -> predicted cancellation_request
  7 true service_complaint      -> predicted visa_entry_question
  5 true out_of_scope           -> predicted rebooking_request


## 2. A scoring head on frozen embeddings: how many labels does it need?
For each budget we draw labeled examples from the training split five times (five seeds) and report mean, spread and range.
Examples are drawn from distinct scenarios, so no two share a situation. "All" is the full training split (100 per route), one run.

In [5]:
def draw(k, seed):
    rng = np.random.default_rng(seed); idx = []
    for ci in range(8):
        rows = [i for i, r in enumerate(tr) if ytr[i] == ci]
        by_scen = {}
        for i in rows: by_scen.setdefault(tr[i]["scenario"], []).append(i)
        scen = rng.permutation(sorted(by_scen))[:k]
        idx += [int(rng.choice(by_scen[s])) for s in scen]
    return np.array(idx)

SEEDS = [1, 2, 3, 4, 5]
curve = {}
for k in (8, 16):
    accs = []
    for s in SEEDS:
        idx = draw(k, s)
        clf = LogisticRegression(C=100, max_iter=2000).fit(E["train"][idx], ytr[idx])
        accs.append((clf.predict(E["test"]) == yte).mean())
    curve[k] = accs
    print(f"{k:2d} per route ({8*k:3d} labels): mean {np.mean(accs):.3f}  sd {np.std(accs, ddof=1):.3f}  range [{min(accs):.3f}, {max(accs):.3f}]")
clf = LogisticRegression(C=100, max_iter=2000).fit(E["train"], ytr)
acc_all = (clf.predict(E["test"]) == yte).mean()
print(f"all (100 per route, {len(ytr)} labels): {acc_all:.3f}")
curve["all"] = [float(acc_all)]
out["head_learning_curve"] = {str(k): [float(a) for a in v] for k, v in curve.items()}

 8 per route ( 64 labels): mean 0.900  sd 0.030  range [0.860, 0.935]


16 per route (128 labels): mean 0.957  sd 0.010  range [0.945, 0.970]
all (100 per route, 800 labels): 0.975


## 3. SetFit against the prototype baseline
**Prototype baseline:** for each route, average the embeddings of its 8 examples; classify a message by the nearest prototype. It trains in milliseconds.
**SetFit:** build positive and negative pairs from the same 8 examples, fine-tune the sentence encoder with a contrastive loss, then fit a logistic head.
Same 8 examples per route, same five seeds, same test split. We also record the training time on this CPU.

In [6]:
def prototype_predict(idx):
    protos = np.stack([E["train"][idx][ytr[idx] == c].mean(0) for c in range(8)])
    protos /= np.linalg.norm(protos, axis=1, keepdims=True)
    return (E["test"] @ protos.T).argmax(1)

proto_acc = []
for s in SEEDS:
    idx = draw(8, s); proto_acc.append((prototype_predict(idx) == yte).mean())
print(f"prototype, 8 per route: mean {np.mean(proto_acc):.3f}  sd {np.std(proto_acc, ddof=1):.3f}  range [{min(proto_acc):.3f}, {max(proto_acc):.3f}]")
out["prototype_8"] = [float(a) for a in proto_acc]

prototype, 8 per route: mean 0.873  sd 0.022  range [0.845, 0.900]


In [7]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
import torch, os
torch.set_num_threads(os.cpu_count())

setfit_acc, setfit_secs = [], []
for s in SEEDS:
    idx = draw(8, s)
    ds = Dataset.from_dict({"text": [tr[i]["text"] for i in idx], "label": [int(ytr[i]) for i in idx]})
    model = SetFitModel.from_pretrained("BAAI/bge-small-en-v1.5")
    args = TrainingArguments(batch_size=16, num_iterations=5, num_epochs=1, seed=s, show_progress_bar=False, report_to="none")
    t = time.perf_counter()
    Trainer(model=model, args=args, train_dataset=ds).train()
    secs = time.perf_counter() - t
    pred = np.array(model.predict([r["text"] for r in te]))
    setfit_acc.append(float((pred == yte).mean())); setfit_secs.append(secs)
    print(f"seed {s}: SetFit acc {setfit_acc[-1]:.3f}  train {secs:5.0f} s   (prototype, same examples: {proto_acc[s-1]:.3f})")

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map: 100%|██████████| 64/64 [00:00<00:00, 4020.06 examples/s]


***** Running training *****


  Num unique pairs = 640


...

  Num unique pairs = 640


  Batch size = 16


  Num epochs = 1


Step,Training Loss
1,0.236000


seed 5: SetFit acc 0.895  train   646 s   (prototype, same examples: 0.870)


In [8]:
d = np.array(setfit_acc) - np.array(proto_acc)
print(f"SetFit  : mean {np.mean(setfit_acc):.3f}  sd {np.std(setfit_acc, ddof=1):.3f}  range [{min(setfit_acc):.3f}, {max(setfit_acc):.3f}]")
print(f"Prototype: mean {np.mean(proto_acc):.3f}  sd {np.std(proto_acc, ddof=1):.3f}  range [{min(proto_acc):.3f}, {max(proto_acc):.3f}]")
print(f"paired difference SetFit - prototype per seed: {np.round(d, 3)}; mean {d.mean():+.3f}; SetFit wins {int((d > 0).sum())} of {len(d)} seeds")
print(f"median training time {np.median(setfit_secs):.0f} s per run on this CPU")
out["setfit_8"] = setfit_acc; out["setfit_train_seconds"] = setfit_secs
out["hardware"] = m.hardware()
(RESULTS / "02_zero_shot_and_setfit.json").write_text(json.dumps(out, indent=2, default=float))

SetFit  : mean 0.919  sd 0.024  range [0.895, 0.950]
Prototype: mean 0.873  sd 0.022  range [0.845, 0.900]
paired difference SetFit - prototype per seed: [0.045 0.05  0.05  0.06  0.025]; mean +0.046; SetFit wins 5 of 5 seeds
median training time 646 s per run on this CPU


1446

### The stronger baseline: a plain logistic-regression head on the same frozen embeddings and the same 8 examples
The prototype is the weakest honest baseline. The head from section 2 uses the same examples with no fine-tuning, so SetFit has to beat that too.

In [9]:
head8 = np.array(curve[8]); sf = np.array(setfit_acc)
d2 = sf - head8
print(f"LR head, 8 per route : mean {head8.mean():.3f}  range [{head8.min():.3f}, {head8.max():.3f}]")
print(f"SetFit, 8 per route  : mean {sf.mean():.3f}  range [{sf.min():.3f}, {sf.max():.3f}]")
print(f"paired difference SetFit - LR head per seed: {np.round(d2, 3)}; mean {d2.mean():+.3f}; SetFit wins {int((d2 > 0).sum())} of {len(d2)} seeds")
out["setfit_minus_lr_head"] = [float(x) for x in d2]
(RESULTS / "02_zero_shot_and_setfit.json").write_text(json.dumps(out, indent=2, default=float))

LR head, 8 per route : mean 0.900  range [0.860, 0.935]
SetFit, 8 per route  : mean 0.919  range [0.895, 0.950]
paired difference SetFit - LR head per seed: [ 0.045  0.02   0.015  0.025 -0.01 ]; mean +0.019; SetFit wins 4 of 5 seeds


1608

## Decision rule
Keep SetFit only if it beats the prototype baseline on the paired seeds by more than the seed spread. With five seeds this is a screen, not a test:
five paired differences cannot establish significance, so a small win is reported as "no clear difference".